# Stage 2 — Preprocessing & Feature Engineering
**Student Performance Analysis**

This notebook demonstrates the preprocessing pipeline built in `src/features/`. Full reasoning for every decision: `reports/stage2_preprocessing_methodology.md`.

Key decisions made this stage:
1. Every predictor individually classified as ordinal (kept as-is) or nominal (one-hot encoded).
2. `course_id` treated as a separate experimental axis: every later model is run **with** and **without** it.
3. `gpa_last_semester` / `gpa_expected_graduation` (the one multicollinear pair, ρ=0.65) — kept both
4. 5-fold stratified CV, fixed with `random_state=42`, saved once so every later model uses identical folds.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.data.load_data import load_raw
from src.features.feature_types import ORDINAL_FEATURES, NOMINAL_FEATURES, COURSE_FEATURE
from src.features.build_features import build_preprocessor, prepare_dataset

df = load_raw()
print(f'Ordinal features: {len(ORDINAL_FEATURES)}')
print(f'Nominal features: {len(NOMINAL_FEATURES)}')
df.head()

## 1. Feature type classification
See `src/features/feature_types.py` for the justification of every single feature - reproduced here as a quick table.

In [ ]:
type_rows = [{'feature': f, 'type': 'ordinal', 'reason': r} for f, r in ORDINAL_FEATURES.items()]
type_rows += [{'feature': f, 'type': 'nominal', 'reason': r} for f, r in NOMINAL_FEATURES.items()]
pd.DataFrame(type_rows).sort_values(['type','feature']).reset_index(drop=True)

## 2. Building both experimental feature sets

In [ ]:
for include_course in (True, False):
    X_raw, y = prepare_dataset(df, include_course=include_course)
    pre = build_preprocessor(include_course=include_course)
    X_enc = pre.fit_transform(X_raw)
    label = 'WITH course_id' if include_course else 'WITHOUT course_id'
    print(f'{label}: raw {X_raw.shape} -> encoded {X_enc.shape}')

## 3. Inspect the encoded column names (sanity check: one-hot vs passthrough)

In [ ]:
X_raw, y = prepare_dataset(df, include_course=True)
pre = build_preprocessor(include_course=True)
pre.fit(X_raw)
feature_names = pre.get_feature_names_out()
print(f'{len(feature_names)} encoded columns')
list(feature_names)

## 4. Fixed stratified CV folds
Generated once by `src/data/make_cv_folds.py`, saved to `data/processed/cv_folds.csv`. Every model in Stages 3-5 loads this file, required for fair comparison.

In [ ]:
cv_folds = pd.read_csv('../data/processed/cv_folds.csv')
print('Fold sizes:'); print(cv_folds['fold'].value_counts().sort_index())
print('\nClass balance per fold:')
pd.crosstab(cv_folds['fold'], cv_folds['grade'])

## Summary

Both feature sets build cleanly (56 columns without course, 65 with). Encoding will be fit **inside** the cross-validation loop in Stage 3 (via `sklearn.Pipeline`), not once on the full dataset, to avoid leakage. Stage 3 will establish baseline models (logistic regression, k-NN) under both feature-set conditions, using the fixed folds loaded above.